# Bitcoin Data Exploration

This notebook explores the Bitcoin dataset and prepares it for RL training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add parent directory to path
sys.path.append('..')

from utils.data_utils import load_data, download_bitcoin_data, validate_data
from features.technical_indicators import add_technical_indicators
from utils.plotting import plot_price_and_indicators

plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)

## Load Bitcoin Data

In [ ]:
# Load Bitcoin data
if os.path.exists('../data/BTC.csv'):
    data = load_data('../data/BTC.csv')
    print("Loaded data from local file")
else:
    print("Downloading Bitcoin data...")
    data = download_bitcoin_data('BTC-USD', '5y', '1d')
    data.to_csv('../data/BTC.csv', index=False)
    print("Data saved to ../data/BTC.csv")

print(f"Data shape: {data.shape}")
data.head()

## Data Quality Assessment

In [ ]:
# Validate data quality
validation_results = validate_data(data)
print("Data Quality Report:")
print(f"Total rows: {validation_results['total_rows']}")
print(f"Data quality score: {validation_results['data_quality_score']}%")
print(f"Missing values: {validation_results['missing_values']}")
print(f"Duplicate rows: {validation_results['duplicate_rows']}")

if validation_results['date_range']:
    print(f"Date range: {validation_results['date_range']['start']} to {validation_results['date_range']['end']}")
    print(f"Total days: {validation_results['date_range']['days']}")

## Basic Statistics

In [ ]:
# Basic statistics
print("Basic Statistics:")
print(data.describe())

# Price statistics
print(f"\nPrice Statistics:")
print(f"Current Price: ${data['close'].iloc[-1]:,.2f}")
print(f"All-time High: ${data['high'].max():,.2f}")
print(f"All-time Low: ${data['low'].min():,.2f}")
print(f"Average Price: ${data['close'].mean():,.2f}")
print(f"Price Volatility: {data['close'].std():,.2f}")

## Price Visualization

In [ ]:
# Plot price over time
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Price chart
axes[0].plot(data.index, data['close'], linewidth=2, color='blue')
axes[0].set_title('Bitcoin Price Over Time', fontsize=16)
axes[0].set_ylabel('Price (USD)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Volume chart
axes[1].bar(data.index, data['volume'], alpha=0.6, color='gray')
axes[1].set_title('Trading Volume', fontsize=16)
axes[1].set_ylabel('Volume', fontsize=12)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Returns Analysis

In [ ]:
# Calculate returns
data['daily_return'] = data['close'].pct_change()
data['log_return'] = np.log(data['close'] / data['close'].shift(1))

# Returns statistics
print("Returns Statistics:")
print(f"Average Daily Return: {data['daily_return'].mean():.4f} ({data['daily_return'].mean()*100:.2f}%)")
print(f"Daily Volatility: {data['daily_return'].std():.4f} ({data['daily_return'].std()*100:.2f}%)")
print(f"Annualized Return: {data['daily_return'].mean() * 252:.4f} ({data['daily_return'].mean() * 252 * 100:.2f}%)")
print(f"Annualized Volatility: {data['daily_return'].std() * np.sqrt(252):.4f} ({data['daily_return'].std() * np.sqrt(252) * 100:.2f}%)")

# Plot returns distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Returns over time
axes[0].plot(data.index, data['daily_return'], alpha=0.7, color='green')
axes[0].set_title('Daily Returns Over Time')
axes[0].set_ylabel('Daily Return')
axes[0].grid(True, alpha=0.3)

# Returns histogram
axes[1].hist(data['daily_return'].dropna(), bins=50, alpha=0.7, color='blue', density=True)
axes[1].set_title('Daily Returns Distribution')
axes[1].set_xlabel('Daily Return')
axes[1].set_ylabel('Density')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Technical Indicators

In [ ]:
# Add technical indicators
data_with_indicators = add_technical_indicators(data.copy())
print(f"Added technical indicators. New shape: {data_with_indicators.shape}")
print(f"New columns: {list(data_with_indicators.columns[len(data.columns):])[:10]}...")  # Show first 10 new columns

In [ ]:
# Plot technical indicators
plot_price_and_indicators(data_with_indicators.tail(252), "Bitcoin Technical Analysis (Last Year)")

## Correlation Analysis

In [ ]:
# Select key indicators for correlation analysis
indicator_cols = ['close', 'volume', 'RSI', 'MACD', 'SMA_20', 'EMA_12', 'BB_width', 'ATR', 'Volatility']
available_cols = [col for col in indicator_cols if col in data_with_indicators.columns]

if len(available_cols) > 1:
    correlation_matrix = data_with_indicators[available_cols].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
                square=True, linewidths=0.5)
    plt.title('Technical Indicators Correlation Matrix')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough indicators available for correlation analysis")

## Data Preparation Summary

In [ ]:
# Final data summary
print("Data Preparation Summary:")
print(f"Original data shape: {data.shape}")
print(f"Data with indicators shape: {data_with_indicators.shape}")
print(f"Missing values after indicators: {data_with_indicators.isnull().sum().sum()}")

# Clean data (remove NaN values)
clean_data = data_with_indicators.dropna()
print(f"Clean data shape: {clean_data.shape}")

# Save processed data
clean_data.to_csv('../data/processed/btc_with_indicators.csv', index=False)
print("Processed data saved to ../data/processed/btc_with_indicators.csv")

print("\nData exploration completed successfully!")